In [ ]:
suppressPackageStartupMessages({
    library(ggplot2)
    library(Seurat)
    library(dplyr)
    library(tidyr)
    library(tibble)
    library(stringr)
    library(dittoSeq)
    library(patchwork)
    library(future)
    plan("multicore", workers = 12)
    options(future.globals.maxSize = 1000 * 1024^5)
    options(stringsAsFactors = FALSE)
    set.seed(123)
})

In [ ]:
path_ref <- '/projects/0/einf2548/cruiz/dmg/data/rna_dmg_atlas_scglue_embbeding.rds'
reference <- readRDS(path_ref)
reference

An object of class Seurat 
19248 features across 397794 samples within 1 assay 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 dimensional reductions calculated: pca, umap

In [ ]:
reference <- AddMetaData(reference, readRDS('../data/dmg_atlas_final_annotation.rds'))

In [ ]:
smart <- readRDS('/projects/0/einf2548/cruiz/dmg/data/rna_dmg_atlas_smart_seq_datasets.rds')
smart

An object of class Seurat 
19283 features across 11767 samples within 3 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 other assays present: prediction.score.coarse, prediction.score.gbmap.based
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
smart <-FindVariableFeatures(smart)%>%ScaleData()%>%RunPCA(verbose = FALSE)%>%
        FindNeighbors(verbose = F) %>% FindClusters(verbose = F)%>%
        RunUMAP(dims = 1:15, reduction = "pca", verbose = F)

Centering and scaling data matrix

Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via the L'Ecuyer-CMRG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”
Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
Found more than one class "dist" in cache; using the first, from namespace 'spam'

Also defined by ‘BiocGenerics’

Found more than one class "dist" in cache; using the first, f

### Reference mapping

In [ ]:
anchors <- FindTransferAnchors(
  reference = reference,
  query = smart,
  features = rownames(reference[["RNA"]]),
  normalization.method = "LogNormalize",
  reference.reduction = "pca",
  dims = 1:50
)

Projecting cell embeddings

Finding neighborhoods

Finding anchors

	Found 6272 anchors



`Map Query` functions run separately

In [ ]:
smart <- TransferData(
  anchorset = anchors, 
  reference = reference,
  query = smart,
  refdata = list(
      dmg.atlas.lvl_1 = 'lvl_1',
      dmg.atlas.lvl_2 = 'lvl_2',
      dmg.atlas.lvl_3 = 'lvl_3',
      dmg.atlas.lvl_4 = 'lvl_4',
      dmg.atlas.lvl_5 = 'lvl_5'
  ),
)

Finding integration vectors

Finding integration vector weights

Predicting cell labels

Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


In [ ]:
smart <- IntegrateEmbeddings(
  anchorset = anchors,
  reference = reference,
  query = smart, 
  new.reduction.name = "ref.pca"
)


Integrating dataset 2 with reference dataset

Finding integration vectors

Integrating data



### UMAP embbeding projection

In [ ]:
smart <- ProjectUMAP(
  query = smart, 
  query.reduction = "ref.pca", 
  reference = reference, 
  reference.reduction = "pca", 
  reduction.model = "umap"
)

Computing nearest neighbors

Running UMAP projection

22:37:58 Read 11767 rows

22:37:58 Processing block 1 of 1

22:37:58 Commencing smooth kNN distance calibration using 12 threads
 with target n_neighbors = 30

22:37:58 Initializing by weighted average of neighbor coordinates using 12 threads

22:37:58 Commencing optimization for 67 epochs, with 353010 positive edges

22:38:01 Finished



In [ ]:
saveRDS(smart, '../data/smartseq2_datasets_pDG_atlas_predictions.rds')